# Reject a status of another type

Reference a status object that is not a declared member of the axis's state type.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from aoa.action_machine.domain import BaseStateEntity, ForeignStateError, Lifecycle, value


class OrderStateEntity(BaseStateEntity):
    """Describe one possible order status."""

    DRAFT = value(code="draft", display_name="Draft")


class OtherStateEntity(BaseStateEntity):
    """A second state type, not the one the axis names."""

    FOREIGN = value(code="foreign", display_name="Foreign")

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
try:
    class BrokenLifecycle(Lifecycle[OrderStateEntity]):
        """Intentionally reference a status outside the axis's type."""

        _template = Lifecycle().state(OrderStateEntity.DRAFT).to(OtherStateEntity.FOREIGN).initial()
except ForeignStateError as error:
    assert "is not a declared status" in str(error)
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("The invalid declaration was accepted")

ForeignStateError: BrokenLifecycle: status 'foreign' (OtherStateEntity) is not a declared status of OrderStateEntity. Allowed: ['draft'].


## What the result means

The template may use only declared members of the axis’s state type. A foreign object does not become a member because its code looks suitable. Refer to the owning type’s declared status when describing its transitions.